# Gaon — train on a FREE Colab GPU

Runtime → Change runtime type → **GPU** (free tier gives a T4).

T4 has no native bf16, so the Colab config uses **fp16**. The model checkpoints
often; to survive disconnects, mount Drive (cell below) and point `out_dir` /
`resume_from` at a Drive path.

Note: a 0.6B model on a 16GB T4 is tight and slow — this proves the free
pipeline end to end. For full-scale training use TRC TPUs or a rented GPU.

In [ ]:
!nvidia-smi -L  # which GPU did Colab give us?

In [ ]:
!git clone https://github.com/k08200/gaon.git
%cd gaon
# torch/numpy already in Colab; install only what's missing
!pip install -q "transformers>=4.51" "datasets>=2.20" "trl>=0.9" pyyaml

In [ ]:
# correctness check (CPU, seconds)
!python scripts/sanity_check.py

### (optional) Persist checkpoints to Google Drive
Run this so a disconnect doesn't lose progress, then edit
`configs/gaon_0.6b_colab.yaml`: set `out_dir: /content/drive/MyDrive/gaon` and
(to continue) `resume_from: /content/drive/MyDrive/gaon/latest.pt`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/gaon

In [ ]:
# download a small data slice (~0.5B tokens, a few minutes)
!python -m src.data.prepare --dataset fineweb-edu --target-tokens 500_000_000 --out data/fineweb_edu

In [ ]:
# train (fp16, single GPU). If OOM: lower micro_batch_size to 2 in the config.
!python -m src.train.train --config configs/gaon_0.6b_colab.yaml

In [ ]:
# sample from the latest checkpoint
!python -m src.eval.generate --ckpt checkpoints/gaon_colab/latest.pt --prompt "The history of" --max-new 80